# CreditWise AI

## Home Credit Default Risk Prediction

### Objective

Develop an AI-powered credit risk assessment system capable of predicting loan default probability using customer financial and behavioral attributes.

### Dataset

Home Credit Default Risk Dataset

### Problem Type

Binary Classification

### Target Variable

- 0 = Loan Repaid
- 1 = Loan Default

## 1. Dataset Loading and Initial Verification

Load the Home Credit training dataset and verify the number of observations and available features.

In [ ]:
import pandas as pd

train_df = pd.read_csv("../data/raw/application_train.csv")

print(train_df.shape)

## 2. Dataset Preview

Inspect a sample of records to understand feature structure and data representation.

In [ ]:
train_df.head()

## 3. Dataset Information

Analyze column data types, non-null counts, and overall dataset structure.

In [ ]:
train_df.info()

## 4. Target Variable Analysis

Examine the distribution of loan repayment and loan default records.

In [ ]:
train_df["TARGET"].value_counts()

## 5. Target Distribution Percentage Analysis

Calculate the percentage of defaulting and non-defaulting customers to assess class imbalance.

In [ ]:
train_df["TARGET"].value_counts(normalize=True) * 100

## 6. Feature Inventory

Review all available dataset features and understand the breadth of information provided.

In [ ]:
train_df.columns.tolist()

## 7. Feature Type Analysis

Determine the number of numerical and categorical features available for modeling.

In [ ]:
train_df.dtypes.value_counts()

## 8. Missing Value Assessment (Absolute Count)

Identify columns containing missing values and quantify the number of missing observations.

In [ ]:
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing.head(20)

## 9. Missing Value Assessment (Percentage)

Evaluate the severity of missing data by calculating the percentage of missing values per feature.

In [ ]:
missing_percent = (train_df.isnull().sum() / len(train_df)) * 100

missing_percent = missing_percent[missing_percent > 0]

missing_percent.sort_values(ascending=False).head(20)

## 10. Target Distribution Visualization

Visualize the imbalance between loan repayment and loan default classes.

In [ ]:
import matplotlib.pyplot as plt

target_counts = train_df["TARGET"].value_counts()

plt.figure(figsize=(6,4))
target_counts.plot(kind="bar")

plt.title("Loan Repayment vs Default Distribution")
plt.xlabel("Target")
plt.ylabel("Count")

plt.show()

## Key Findings

### Dataset Characteristics

- Total Records: 307,511
- Total Features: 122
- Numerical Features: 106
- Categorical Features: 16

### Target Distribution

- Loan Repaid (0): 91.93%
- Loan Default (1): 8.07%

### Initial Observations

- The dataset exhibits significant class imbalance.
- Accuracy alone will not be an appropriate evaluation metric.
- Several housing-related features contain more than 60% missing values.
- Feature selection and missing value treatment will be critical preprocessing steps.
- Metrics such as ROC-AUC, Precision, Recall, and F1-score will be considered during model evaluation.

# Feature Quality Assessment

The objective of this phase is to evaluate feature usefulness, identify low-quality columns, and develop a data preprocessing strategy for model training.

## High Missing Value Feature Identification

Features with extremely high missingness can negatively affect model quality and increase preprocessing complexity.

This analysis identifies features with more than 60% missing values for further evaluation.

In [ ]:
high_missing = missing_percent[missing_percent > 60]

print(f"Number of features with >60% missing values: {len(high_missing)}")

high_missing.sort_values(ascending=False)

## Feature Retention Assessment

Estimate the number of features that would remain if highly incomplete columns were removed.

This helps evaluate whether aggressive feature elimination would significantly reduce the information available for modeling.

In [ ]:
total_features = train_df.shape[1] - 1   #excluding TARGET

remaining_features = total_features - len(high_missing)

print("Total Features:", total_features)
print("Features >60% Missing:", len(high_missing))
print("Remaining Features:", remaining_features)

## Categorical Feature Identification

Machine learning algorithms require categorical variables to be transformed into numerical representations before model training.

This analysis identifies all categorical features present in the dataset and provides an overview of the variables that will require encoding during preprocessing.

In [ ]:
categorical_cols = train_df.select_dtypes(include=["object", "string"]).columns.tolist()

print("Number of categorical features:", len(categorical_cols))
categorical_cols

## Categorical Feature Cardinality Analysis

Cardinality refers to the number of unique values present within a categorical feature.

Understanding feature cardinality is important because it influences the choice of encoding strategy. Low-cardinality features are often suitable for one-hot encoding, while high-cardinality features may require alternative approaches to avoid excessive dimensionality.

In [ ]:
for col in categorical_cols:
    print(f"{col}: {train_df[col].nunique()}")

## Observation: Categorical Feature Cardinality

Most categorical features in the dataset exhibit low cardinality, with fewer than 20 unique values. Such features can typically be encoded efficiently using one-hot encoding without significantly increasing dimensionality.

One feature, `ORGANIZATION_TYPE`, contains 58 unique categories and represents a relatively high-cardinality variable. Applying one-hot encoding directly to this feature may introduce a large number of additional columns and increase model complexity.

During the preprocessing phase, low-cardinality features will be considered for one-hot encoding, while alternative encoding techniques such as frequency encoding will be evaluated for `ORGANIZATION_TYPE`.


# Statistical Feature Analysis

The objective of this phase is to investigate relationships between individual features and loan default behavior.

Understanding these relationships helps identify predictive signals, guide feature engineering decisions, and improve model interpretability.

## Missingness Signal Analysis: Vehicle Age

Missing values are not always random.

This analysis investigates whether the absence of vehicle age information is associated with different loan default behavior.

If default rates differ significantly, missingness itself may become a useful predictive feature.

In [ ]:
train_df["OWN_CAR_AGE_MISSING"] = train_df["OWN_CAR_AGE"].isnull().astype(int)

train_df.groupby("OWN_CAR_AGE_MISSING")["TARGET"].mean()

## Income Analysis by Default Status

Income level is one of the most important factors in credit risk assessment.

This analysis compares average applicant income between customers who repaid loans and those who defaulted.

In [ ]:
train_df.groupby("TARGET")["AMT_INCOME_TOTAL"].mean()

## Income Distribution Visualization

Visualize the distribution of applicant income across repayment outcomes.

The objective is to identify potential differences, outliers, and income-related risk patterns.

In [ ]:
import matplotlib.pyplot as plt

train_df.boxplot(
    column="AMT_INCOME_TOTAL",
    by="TARGET",
    figsize=(8,5)
)

plt.title("Income Distribution by Default Status")
plt.suptitle("")
plt.show()

## Credit Amount Analysis

Loan size can influence repayment risk.

This analysis compares average credit amounts between defaulting and non-defaulting customers.

In [ ]:
train_df.groupby("TARGET")["AMT_CREDIT"].mean()

## Credit Amount Distribution Visualization

Visualize the distribution of requested credit amounts across repayment outcomes.

This helps identify whether larger or smaller loans are associated with higher default risk.

In [ ]:
train_df.boxplot(
    column="AMT_CREDIT",
    by="TARGET",
    figsize=(8,5)
)

plt.title("Credit Amount by Default Status")
plt.suptitle("")
plt.show()

## Gender-Based Default Analysis

Evaluate whether default behavior differs across applicant gender categories.

This analysis provides an initial understanding of demographic risk patterns within the dataset.

In [ ]:
pd.crosstab(
    train_df["CODE_GENDER"],
    train_df["TARGET"],
    normalize="index"
)

## Education Level and Default Risk

Educational attainment often correlates with income stability and employment opportunities.

This analysis investigates how default rates vary across education categories.

In [ ]:
pd.crosstab(
    train_df["NAME_EDUCATION_TYPE"],
    train_df["TARGET"],
    normalize="index"
).sort_values(1, ascending=False)

## Income Source and Default Risk

Income source may influence repayment reliability.

This analysis compares default rates across different employment and income categories to identify higher-risk applicant groups.

In [ ]:
pd.crosstab(
    train_df["NAME_INCOME_TYPE"],
    train_df["TARGET"],
    normalize="index"
).sort_values(1, ascending=False)

## Statistical Analysis Findings

### Missing Value Signal

Applicants with missing vehicle-age information exhibited a higher default rate than applicants with available vehicle-age data. This suggests that missingness itself may contain predictive information and should be considered during feature engineering.

### Income and Credit Amount

Average income and average credit amount showed only modest differences between repayment and default groups. These features may still contribute predictive value when combined with other variables.

### Gender

Male applicants exhibited a higher default rate than female applicants, indicating potential predictive value.

### Education

Default rates decreased consistently with increasing education level. Applicants with higher educational attainment demonstrated substantially lower default risk.

### Income Type

Default behavior varied across income categories, suggesting that employment and income source are important risk indicators.

### Preliminary Conclusion

Education level, income type, gender, and missing-value indicators appear to be promising predictive features and will be retained for future modeling and feature engineering.

# Feature Engineering and Correlation Analysis

The objective of this phase is to identify highly predictive features, evaluate relationships with the target variable, and design feature engineering strategies that improve model performance.

The resulting feature set will form the basis of the machine learning pipeline.

## External Credit Score Features

The dataset contains externally generated credit-related scores represented by EXT_SOURCE_1, EXT_SOURCE_2, and EXT_SOURCE_3.

These variables are expected to carry significant predictive information and are evaluated separately.

In [ ]:
ext_features = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

train_df[ext_features + ["TARGET"]].corr()["TARGET"].sort_values()

## Missing Value Assessment: External Credit Scores

Before feature engineering, the completeness of external credit score variables is evaluated to determine whether imputation or alternative treatment strategies are required.

In [ ]:
train_df[ext_features].isnull().mean() * 100

## Distribution Analysis: EXT_SOURCE_2

Visualize the relationship between EXT_SOURCE_2 and loan default outcomes.

In [ ]:
train_df.boxplot(
    column="EXT_SOURCE_2",
    by="TARGET",
    figsize=(8,5)
)

plt.title("EXT_SOURCE_2 by Default Status")
plt.suptitle("")
plt.show()

## Applicant Age Analysis

Applicant age is derived from DAYS_BIRTH and evaluated as a potential predictor of credit risk.

In [ ]:
train_df["AGE_YEARS"] = abs(train_df["DAYS_BIRTH"]) / 365

train_df.groupby("TARGET")["AGE_YEARS"].mean()

## Age Distribution by Default Status

Compare age distributions between defaulting and non-defaulting customers.

In [ ]:
train_df.boxplot(
    column="AGE_YEARS",
    by="TARGET",
    figsize=(8,5)
)

plt.title("Age by Default Status")
plt.suptitle("")
plt.show()

## Employment History Analysis

Employment duration is derived from DAYS_EMPLOYED and evaluated as a potential indicator of repayment stability.

In [ ]:
train_df["EMPLOYMENT_YEARS"] = abs(train_df["DAYS_EMPLOYED"]) / 365

train_df.groupby("TARGET")["EMPLOYMENT_YEARS"].mean()

## Numerical Feature Correlation Analysis

Identify numerical variables that exhibit the strongest relationships with the target variable.

These features are potential candidates for model development and feature engineering.

In [ ]:
correlations = train_df.corr(numeric_only=True)["TARGET"]

correlations.sort_values().head(20)

In [ ]:
correlations.sort_values(ascending=False).head(20)

## Correlation Analysis Findings

### External Credit Score Features

The strongest relationships with loan default risk were observed for the external credit score variables:

* EXT_SOURCE_3 (-0.1789)
* EXT_SOURCE_2 (-0.1605)
* EXT_SOURCE_1 (-0.1553)

All three variables exhibited negative correlations with the target variable, indicating that applicants with higher external credit scores are less likely to default. These features represent some of the most important predictors identified so far and will be retained for model development.

### Missing Value Assessment of External Credit Scores

The external credit score features exhibited varying levels of missing data:

* EXT_SOURCE_1: 56.38% missing
* EXT_SOURCE_2: 0.21% missing
* EXT_SOURCE_3: 19.83% missing

Despite substantial missingness in EXT_SOURCE_1 and EXT_SOURCE_3, both variables demonstrated strong predictive relationships with loan default behavior. Therefore, these features will be retained and appropriate imputation strategies will be applied during preprocessing.

### Applicant Age Analysis

Applicant age was derived from the DAYS_BIRTH feature and showed a meaningful relationship with default risk.

Average age by repayment outcome:

* Non-defaulting applicants: 44.21 years
* Defaulting applicants: 40.78 years

This observation suggests that younger applicants are generally associated with higher default risk, making age an important candidate for feature engineering.

### Employment Duration Analysis

Initial employment-duration analysis revealed unrealistic average employment lengths exceeding one hundred years.

This indicates the presence of placeholder or anomalous values within the DAYS_EMPLOYED feature. Additional data-quality assessment is required before employment-related features can be used for model training.

### Numerical Correlation Analysis

Among numerical variables, the strongest predictors identified so far include:

* EXT_SOURCE_1
* EXT_SOURCE_2
* EXT_SOURCE_3
* Applicant Age (AGE_YEARS)
* Employment Duration (EMPLOYMENT_YEARS)

The analysis indicates that external credit information, demographic characteristics, and employment history are likely to play a significant role in predicting loan default behavior.

### Preliminary Conclusion

The correlation analysis confirms that external credit scores are the most informative features discovered so far. Age also demonstrates meaningful predictive value, while employment-related variables require further preprocessing due to data-quality issues.

These findings will guide subsequent feature-engineering, missing-value treatment, and model-development activities.


## Employment Duration Data Quality Assessment

Initial analysis revealed unrealistic employment durations caused by placeholder values in the DAYS_EMPLOYED feature.

These values will be investigated and corrected before feature engineering.

In [ ]:
(train_df["DAYS_EMPLOYED"] == 365243).sum()

In [ ]:
(
    (train_df["DAYS_EMPLOYED"] == 365243)
    .value_counts(normalize=True)
    * 100
)

In [ ]:
train_df.groupby(
    train_df["DAYS_EMPLOYED"] == 365243
)["TARGET"].mean()

## Feature Engineering Candidates

Based on exploratory and statistical analysis, several engineered features have been identified as promising candidates for model development.

In [ ]:
train_df["EXT_SOURCE_MEAN"] = train_df[
    ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
].mean(axis=1)

In [ ]:
train_df[
    ["EXT_SOURCE_MEAN", "TARGET"]
].corr()["TARGET"]

## Feature Engineering Findings

### Employment Duration Placeholder Values

Analysis of the DAYS_EMPLOYED feature revealed that 55,374 records (approximately 18% of the dataset) contain a placeholder value of 365243 days.

These records exhibited a lower default rate than applicants with standard employment records, suggesting that the placeholder values may contain useful information rather than representing random missing data.

The placeholder values will therefore be treated as missing data while preserving their informational value through a dedicated indicator feature.

### External Credit Score Aggregation

A new feature, EXT_SOURCE_MEAN, was created by averaging EXT_SOURCE_1, EXT_SOURCE_2, and EXT_SOURCE_3.

The engineered feature achieved a correlation of -0.222 with the target variable, exceeding the predictive strength of each individual external credit score feature.

This result suggests that combining multiple external credit indicators provides a stronger representation of applicant creditworthiness and should improve model performance.

### Features with Strong Evidence of Predictive Value

The following variables demonstrated clear relationships with loan default behavior through statistical analysis and correlation assessment:

* EXT_SOURCE_1
* EXT_SOURCE_2
* EXT_SOURCE_3
* EXT_SOURCE_MEAN
* AGE_YEARS
* NAME_EDUCATION_TYPE
* NAME_INCOME_TYPE
* OWN_CAR_AGE_MISSING

These features will be prioritized during model development and feature engineering.

### Features Requiring Further Validation

The following variables exhibited weaker or indirect evidence of predictive value and require further evaluation through machine learning models:

* CODE_GENDER
* AMT_INCOME_TOTAL
* AMT_CREDIT
* EMPLOYMENT_YEARS

Although preliminary analysis suggests potential usefulness, their final contribution should be assessed using feature importance and explainability techniques.

### Preliminary Conclusion

The analyses conducted so far indicate that external credit information, demographic characteristics, education level, income source, and missing-value indicators are likely to play an important role in predicting loan default risk.

Additional feature engineering and model-based evaluation will be performed to validate these findings and identify the most influential predictors.


# Phase Summary

At this stage, exploratory data analysis, feature quality assessment, statistical analysis, correlation analysis, and initial feature engineering have been completed.

### Key Findings

* The dataset exhibits significant class imbalance, with non-default cases substantially outnumbering default cases.
* Several property-related features contain high levels of missing data and require careful evaluation before removal.
* External credit score variables (EXT_SOURCE_1, EXT_SOURCE_2, and EXT_SOURCE_3) demonstrated the strongest relationships with loan default risk.
* The engineered feature EXT_SOURCE_MEAN showed stronger predictive potential than any individual external credit score feature.
* Younger applicants exhibited higher default rates than older applicants.
* Education level and income type showed meaningful relationships with repayment behavior.
* Missing-value indicators demonstrated predictive potential, suggesting that missingness itself may contain useful information.
* Employment-related variables contain placeholder values that require preprocessing before model development.

### Current Status

The project has successfully identified several promising predictive features and established a preliminary feature-engineering strategy.

The next phase focuses on designing a production-ready preprocessing pipeline, including missing-value treatment, categorical encoding, feature transformation, and preparation of the final training dataset for machine learning models.


# Preprocessing Strategy Design

The objective of this phase is to define a structured preprocessing pipeline for machine learning model development.

This includes missing-value treatment, feature engineering, categorical encoding, numerical feature preparation, and feature selection decisions.

The resulting preprocessing strategy will be used to construct the final training dataset for model development.

## Feature Treatment Framework

Each feature will be assigned to one of the following preprocessing categories:

- Retain without modification
- Retain with imputation
- Retain with engineered indicators
- Encode categorical values
- Transform into engineered features
- Remove from the training dataset

These decisions are based on exploratory analysis, statistical findings, and predictive potential.

In [ ]:
high_missing.index.tolist()

In [ ]:
important_features = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AGE_YEARS",
    "OWN_CAR_AGE"
]

(train_df[important_features].isnull().mean() * 100).sort_values(
    ascending=False
)

In [ ]:
for col in categorical_cols:
    missing_pct = train_df[col].isnull().mean() * 100

    print(f"{col}: {missing_pct:.2f}%")

## Preprocessing Strategy Decisions

Initial preprocessing decisions were established based on feature quality, missing-value analysis, correlation assessment, and business relevance.

### Features to Retain

- EXT_SOURCE_1
- EXT_SOURCE_2
- EXT_SOURCE_3
- EXT_SOURCE_MEAN
- AGE_YEARS
- AMT_INCOME_TOTAL
- AMT_CREDIT
- AMT_ANNUITY
- NAME_EDUCATION_TYPE
- NAME_INCOME_TYPE
- OCCUPATION_TYPE
- OWN_CAR_AGE
- OWN_CAR_AGE_MISSING

### Features Requiring Imputation

- EXT_SOURCE_1
- EXT_SOURCE_2
- EXT_SOURCE_3
- AMT_ANNUITY
- OWN_CAR_AGE

### Features Requiring Missing Indicators

- OWN_CAR_AGE
- EXT_SOURCE_1
- DAYS_EMPLOYED placeholder values

### Candidate Features for Removal

- YEARS_BUILD_AVG
- YEARS_BUILD_MEDI
- YEARS_BUILD_MODE
- COMMONAREA_AVG
- COMMONAREA_MEDI
- COMMONAREA_MODE
- FLOORSMIN_AVG
- FLOORSMIN_MEDI
- FLOORSMIN_MODE
- LIVINGAPARTMENTS_AVG
- LIVINGAPARTMENTS_MEDI
- LIVINGAPARTMENTS_MODE
- NONLIVINGAPARTMENTS_AVG
- NONLIVINGAPARTMENTS_MEDI
- NONLIVINGAPARTMENTS_MODE
- FONDKAPREMONT_MODE
- HOUSETYPE_MODE
- WALLSMATERIAL_MODE
- EMERGENCYSTATE_MODE

## Categorical Feature Encoding Assessment

Categorical variables require transformation before they can be used by machine learning algorithms.

The objective of this analysis is to evaluate the number of unique categories within each categorical feature and determine an appropriate encoding strategy.

Features with a small number of categories may be suitable for one-hot encoding, while features with high cardinality may require alternative approaches such as frequency encoding or target encoding.

This assessment will guide the final preprocessing pipeline and help balance model performance, interpretability, and computational efficiency.

In [ ]:
for col in categorical_cols:
    print(f"\n{col}")
    print(train_df[col].nunique())

In [ ]:
train_df["ORGANIZATION_TYPE"].nunique()

## Categorical Encoding Strategy

Categorical features were evaluated based on cardinality, missing-value patterns, and business relevance.

### One-Hot Encoding

The following features contain a relatively small number of categories and are suitable for one-hot encoding:

- NAME_CONTRACT_TYPE
- CODE_GENDER
- FLAG_OWN_CAR
- FLAG_OWN_REALTY
- NAME_TYPE_SUITE
- NAME_INCOME_TYPE
- NAME_EDUCATION_TYPE
- NAME_FAMILY_STATUS
- NAME_HOUSING_TYPE
- OCCUPATION_TYPE
- WEEKDAY_APPR_PROCESS_START

For OCCUPATION_TYPE, missing values will first be replaced with an "Unknown" category before encoding.

### Frequency Encoding

ORGANIZATION_TYPE contains 58 unique categories and exhibits high cardinality.

To avoid excessive dimensionality while preserving potentially useful information, frequency encoding will be applied.

### Features Marked for Removal

The following categorical variables contain substantial missing data and limited business value and are therefore candidates for removal:

- FONDKAPREMONT_MODE
- HOUSETYPE_MODE
- WALLSMATERIAL_MODE
- EMERGENCYSTATE_MODE

### Conclusion

The selected encoding strategy balances interpretability, predictive potential, and computational efficiency while minimizing unnecessary feature expansion.

# Final Feature Treatment Plan

## Feature Treatment Table

| Feature / Feature Group | Treatment Strategy | Reason |
|----------|----------|----------|
| TARGET | Keep | Prediction target variable |
| EXT_SOURCE_1 | Median Imputation + Missing Indicator | Strong predictive power, high missingness |
| EXT_SOURCE_2 | Median Imputation | Strong predictive power, negligible missingness |
| EXT_SOURCE_3 | Median Imputation + Missing Indicator | Strong predictive power, moderate missingness |
| EXT_SOURCE_MEAN | Keep | Strongest correlation discovered during analysis |
| AGE_YEARS | Keep | Engineered feature with predictive value |
| DAYS_BIRTH | Drop after feature creation | Replaced by AGE_YEARS |
| DAYS_EMPLOYED | Replace placeholder values and engineer new feature | Contains invalid placeholder values |
| EMPLOYMENT_YEARS | Under Evaluation | Requires placeholder-value correction before final decision |
| DAYS_EMPLOYED_PLACEHOLDER | Create Indicator Feature | Placeholder values contain useful information |
| AMT_INCOME_TOTAL | Keep | Core financial feature |
| AMT_CREDIT | Keep | Core financial feature |
| AMT_ANNUITY | Median Imputation | Important financial feature with minimal missingness |
| OWN_CAR_AGE | Median Imputation | Potentially useful numeric feature |
| OWN_CAR_AGE_MISSING | Keep | Missingness demonstrated predictive value |
| NAME_CONTRACT_TYPE | One-Hot Encode | Low cardinality |
| CODE_GENDER | One-Hot Encode | Low cardinality |
| FLAG_OWN_CAR | One-Hot Encode | Binary feature |
| FLAG_OWN_REALTY | One-Hot Encode | Binary feature |
| NAME_TYPE_SUITE | Mode Imputation + One-Hot Encode | Low missingness |
| NAME_INCOME_TYPE | One-Hot Encode | Business-relevant feature |
| NAME_EDUCATION_TYPE | One-Hot Encode | Demonstrated predictive value |
| NAME_FAMILY_STATUS | One-Hot Encode | Low cardinality |
| NAME_HOUSING_TYPE | One-Hot Encode | Low cardinality |
| OCCUPATION_TYPE | Fill "Unknown" + One-Hot Encode | Business-relevant feature |
| WEEKDAY_APPR_PROCESS_START | One-Hot Encode | Low cardinality |
| ORGANIZATION_TYPE | Frequency Encoding | High cardinality (58 categories) |
| YEARS_BUILD_* | Drop | Excessive missingness |
| COMMONAREA_* | Drop | Excessive missingness |
| FLOORSMIN_* | Drop | Excessive missingness |
| LIVINGAPARTMENTS_* | Drop | Excessive missingness |
| NONLIVINGAPARTMENTS_* | Drop | Excessive missingness |
| FONDKAPREMONT_MODE | Drop | High missingness |
| HOUSETYPE_MODE | Drop | High missingness |
| WALLSMATERIAL_MODE | Drop | High missingness |
| EMERGENCYSTATE_MODE | Drop | High missingness |